In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.append(str(Path.cwd().parent / "src"))
from features import FEATURES
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from xgboost import XGBClassifier

PROJ = Path.home() / "Documents/v2x-ids"
df = pd.read_parquet(PROJ / "data/processed/features_run79200.parquet")
X, y, groups = df[FEATURES], df["is_attack"], df["sender"]

def best_threshold(col, tr):
    vals, yt = df.iloc[tr][col], y.iloc[tr]
    best_f1, best_t = 0, None
    for t in np.unique(np.quantile(vals.dropna(), np.linspace(0.5, 0.995, 100))):
        f = f1_score(yt, (vals > t).fillna(False).astype(int))
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t

def scores(model, fold, yt, yp):
    return {"model": model, "fold": fold,
            "precision": precision_score(yt, yp), "recall": recall_score(yt, yp),
            "f1": f1_score(yt, yp),
            "false_alarm": ((yp == 1) & (yt == 0)).sum() / (yt == 0).sum()}

In [4]:
rows, per_attack = [], []
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
t0 = time.time()

for fold, (tr, te) in enumerate(sgkf.split(X, df["label"], groups), start=1):
    test = df.iloc[te]
    yt = y.iloc[te].to_numpy()

    # Rule baseline (threshold from this fold's training data)
    ssc = (test["speed_mismatch"] > best_threshold("speed_mismatch", tr)).fillna(False).astype(int).to_numpy()
    rows.append(scores("Rule SSC", fold, yt, ssc))

    # Random Forest
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced")
    rf.fit(X.iloc[tr], y.iloc[tr])
    p_rf = rf.predict(X.iloc[te])
    rows.append(scores("Random Forest", fold, yt, p_rf))

    # XGBoost
    ratio = (y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum()
    xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                        tree_method="hist", scale_pos_weight=ratio, random_state=42)
    xgb.fit(X.iloc[tr], y.iloc[tr])
    rows.append(scores("XGBoost", fold, yt, xgb.predict(X.iloc[te])))

    # Per-attack detection rate for Random Forest
    pa = test.assign(pred=p_rf)[test.is_attack == 1].groupby("attack_name")["pred"].mean()
    per_attack.append(pa.rename(f"fold{fold}"))
    print(f"fold {fold} done ({time.time() - t0:.0f} s)")

res = pd.DataFrame(rows)

fold 1 done (9 s)
fold 2 done (20 s)
fold 3 done (30 s)
fold 4 done (40 s)
fold 5 done (51 s)


In [5]:
summary = res.groupby("model")[["precision", "recall", "f1", "false_alarm"]].agg(["mean", "std"]).round(3)
summary

precision        recall            f1        false_alarm       
                   mean    std   mean    std   mean    std        mean    std
model                                                                        
Random Forest     0.945  0.020  0.878  0.023  0.910  0.018       0.036  0.013
Rule SSC          0.954  0.009  0.635  0.033  0.762  0.025       0.021  0.003
XGBoost           0.936  0.018  0.884  0.024  0.909  0.016       0.043  0.012

In [6]:
pa = pd.concat(per_attack, axis=1)
attack_table = pd.DataFrame({
    "detected_%": (pa.mean(axis=1) * 100).round(1),
    "std_%": (pa.std(axis=1) * 100).round(1),
}).sort_values("detected_%")
attack_table

,detected_%,std_%
attack_name,,
ConstPosOffset,3.2,2.2
DataReplaySybil,54.1,8.9
DoSDisruptiveSybil,68.2,11.9
GridSybil,70.0,15.6
EventualStop,77.3,23.3
DataReplay,78.7,8.1
RandomPosOffset,92.0,2.7
Disruptive,96.0,2.3
DelayedMessages,98.1,1.6


In [7]:
thesis = res.groupby("model")[["precision", "recall", "f1", "false_alarm"]].agg(
    lambda s: f"{s.mean():.3f} ± {s.std():.3f}")
thesis.to_csv(PROJ / "results/task11_crossval_summary.csv")
attack_table.to_csv(PROJ / "results/task11_per_attack.csv")
res.to_csv(PROJ / "results/task11_all_folds.csv", index=False)
thesis

,precision,recall,f1,false_alarm
model,,,,
Random Forest,0.945 ± 0.020,0.878 ± 0.023,0.910 ± 0.018,0.036 ± 0.013
Rule SSC,0.954 ± 0.009,0.635 ± 0.033,0.762 ± 0.025,0.021 ± 0.003
XGBoost,0.936 ± 0.018,0.884 ± 0.024,0.909 ± 0.016,0.043 ± 0.012
